[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/27_cross_attention.ipynb)

# 🟡 Medium: Multi-Head Cross-Attention

*Attention & Transformers*
Implement **multi-head cross-attention** as a `flax.nnx.Module`: the queries come
from one sequence, the keys and values from a different one, and the two have
unrelated lengths.

$$Q = x_q W_q,\quad K = x_{kv} W_k,\quad V = x_{kv} W_v,\qquad
\text{out} = \big[\operatorname{softmax}\!\big(\tfrac{Q_hK_h^\top}{\sqrt{d_h}} + M\big)V_h\big]_{h}W_o$$

### Signature
- `MultiHeadCrossAttention(d_model, num_heads, *, d_context=None, rngs: nnx.Rngs)`
- `d_context` defaults to `d_model`
- `__call__(x_q, x_kv, mask=None)` maps
  `(B, T_q, d_model)`, `(B, T_kv, d_context)` `->` `(B, T_q, d_model)`
- `mask` is boolean, broadcastable to `(B, H, T_q, T_kv)`; `True` = attend.
  A key-padding mask is `(B, 1, 1, T_kv)`

### Rules
- Subclass `nnx.Module`; no `nnx.MultiHeadAttention`, no
  `jax.nn.dot_product_attention`
- Parameters named exactly `w_q`, `w_k`, `w_v`, `w_o`, all `nnx.Param`, no biases:
  - `w_q`: `(d_model, d_model)`
  - `w_k`, `w_v`: `(d_context, d_model)`
  - `w_o`: `(d_model, d_model)`
- Initialise each with `jax.random.normal(rngs.params(), shape) / sqrt(fan_in)`
- **No causal mask** — the whole context is visible to every query
- `T_q` and `T_kv` are independent; nothing may assume they match

### Where this actually shows up
- **Encoder–decoder** (the original transformer, T5, Whisper): decoder tokens
  query the encoder's finished representation. $T_q$ is the tokens generated so
  far, $T_{kv}$ the source sentence or the audio frames.
- **Diffusion** (Stable Diffusion and descendants): the UNet's image latents are
  the queries and CLIP text embeddings are the keys/values. This is the *only*
  place the prompt enters the network — swap the K/V stream and you swap the
  prompt. It is also why `d_context` is a separate number: text encoders are 768
  or 1024 wide, UNet blocks are 320/640/1280.
- **Perceiver / Flamingo / DETR**: a small learned latent array of size $T_q = 64$
  queries an enormous input of size $T_{kv} = 50000$. Cost is
  $O(T_q T_{kv})$, not $O(T_{kv}^2)$ — cross-attention is how you get a
  fixed-cost bottleneck onto arbitrarily large inputs.

### The structural property worth naming in an interview
Cross-attention **cannot mix information across query positions**. Output $i$
depends on $x_q[i]$ and on all of $x_{kv}$, and on no other query. It is a
per-query lookup into a shared memory — batched, not sequential. That is why a
decoder block is always *self-attention, then cross-attention, then MLP*: the
self-attention layer is what lets query positions talk to each other, and
removing it leaves a model that can never build a representation spanning two
output tokens.

The second consequence is that the two streams get **separate lengths and
separate masks**. The causal mask belongs to self-attention; what cross-attention
needs is a key-padding mask over $T_{kv}$, shaped `(B, 1, 1, T_kv)` so it
broadcasts across heads and queries. Writing a `(T, T)` mask here is a bug that
only shows up when the two sequences happen to differ in length.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp
from flax import nnx

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp
from flax import nnx


class MultiHeadCrossAttention(nnx.Module):
    """Queries from x_q, keys/values from x_kv. (B, T_q, d_model) out."""

    def __init__(self, d_model: int, num_heads: int, *, d_context: int = None,
                 rngs: nnx.Rngs):
        pass  # Replace this

    def __call__(self, x_q, x_kv, mask=None):
        """x_q: (B, T_q, d_model), x_kv: (B, T_kv, d_context)."""
        pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax
import jax.numpy as jnp
from flax import nnx

# A diffusion-style bridge: 256 image latents query 77 CLIP text tokens.
attn = MultiHeadCrossAttention(d_model=320, num_heads=8, d_context=768,
                               rngs=nnx.Rngs(params=0))
latents = jax.random.normal(jax.random.key(0), (1, 256, 320))
text = jax.random.normal(jax.random.key(1), (1, 77, 768))
print("w_q", attn.w_q.shape, " w_k", attn.w_k.shape, " out", attn(latents, text).shape)

# Query positions never interact: perturbing one query leaves the others alone.
small = MultiHeadCrossAttention(d_model=16, num_heads=2, rngs=nnx.Rngs(params=0))
xq = jax.random.normal(jax.random.key(2), (1, 4, 16))
xkv = jax.random.normal(jax.random.key(3), (1, 6, 16))
base = small(xq, xkv)
poked = small(xq.at[:, 2].set(99.0), xkv)
print("max change at query 0:", float(jnp.abs(base[:, 0] - poked[:, 0]).max()))
print("max change at query 2:", float(jnp.abs(base[:, 2] - poked[:, 2]).max()))

# A key-padding mask hiding the last two context tokens.
pad = jnp.array([[[[True, True, True, True, False, False]]]])   # (1, 1, 1, 6)
print("masked out:", small(xq, xkv, pad).shape)

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("cross_attention")

# hint("cross_attention")      # stuck? nudge without the answer
# solution("cross_attention")  # spoiler: the reference implementation